# LAB 2 — Do texto e do número à evidência

**Disciplina** Tomada de Decisão Baseada em Dados para Gestores de Segurança Pública (CAES-PMERJ / FGV)
**Aula** 2 — Dados, IA e decisões com probabilidade (5 de novembro de 2026, presencial)
**Duração** Lab 1 das 10:00 às 11:45 (Partes 1 a 4) · Lab 2 das 14:45 às 16:30 (Partes 5 a 8)
**Entrega** Este notebook preenchido até a checagem final (Parte 8), antes da aula 3

> **Para quem é este lab:** gestores que decidem. Você não precisa programar. As células de cálculo vêm prontas, quase todas em SQL; o grupo troca `'TODO'` pela resposta, entre aspas simples, e aperta Shift+Enter. Cada resposta fica gravada numa tabela do espaço `blue-blood`. O que importa é **interpretar** o resultado e **decidir**.

## Antes de começar

1. Importe no Databricks Free Edition: **Workspace → (seu usuário) → Import → File → `lab-2.ipynb`**.
2. Conecte ao compute *Serverless* e rode as células **na ordem**, de cima para baixo. A primeira célula de código abre o espaço `blue-blood` criado no lab 1 (ou cria, se este workspace ainda não tiver).
3. Se a internet do laboratório (ou a do Databricks Free Edition, que é restrita) falhar, o notebook grava no volume **dados reais embutidos** (12 narrativas e a série do ISP das CISPs da RISP 1 e das CISPs 33 e 64). O lab continua funcionando.

**O cenário.** A **Blue Blood** é uma força policial fictícia cuja área coincide com o estado do Rio de Janeiro: Comando de Área = RISP, batalhão = AISP, companhia = CISP. Organização, pessoas e números internos são fictícios; os dados de criminalidade são reais e públicos (ISP-RJ).

**Onde fica cada coisa.** Arquivos baixados vão para o volume `bronze.arquivos`; tabelas, para as zonas `bronze`, `silver` e `gold`. Tudo o que o grupo escreve ou decide vira uma tabela `gold.lab2_...`.

**Onde este lab está no arco.**

| Etapa | Neste lab |
| --- | --- |
| 2. Critérios | Julgamento com probabilidade e limiar, com `ai_decide` (Parte 6) |
| 3. Coleta e armazenamento | Narrativas e série do ISP na Bronze; anonimização antes de compartilhar (Partes 1 e 4) |
| 4. Processamento e visualização | Classificar, agrupar, comparar com a base certa e em taxa (Partes 2, 3 e 5) |
| 5. Decisão com critérios e risco | Aplicar o limiar e escolher o que levar para a aula 3 (Partes 6 e 7) |

**Fontes.** Narrativas: ALMEIDA, M. G. de; BOTEGA, L. C.; LIMA, P. M. de P. *Local Anonymization of Free-Text Robbery Police Reports in Brazilian Portuguese*, KDMiLe/SBC, 2026 — dataset CC BY 4.0, [repositório](https://github.com/murilogoes/bo-anonymizer-ptbr). Séries: [ISPDados](https://www.ispdados.rj.gov.br/), Instituto de Segurança Pública do RJ.

## Objetivos de aprendizagem

1. **Classificar** narrativas de boletins com regras e **auditar** o resultado à mão.
2. **Interpretar** grupos que a máquina encontra sozinha em textos.
3. **Medir** quanto dado pessoal um método de anonimização deixa vazar e decidir o que pode ser compartilhado.
4. **Comparar** uma série do ISP com a base certa e **normalizar** por população.
5. **Usar** um julgamento com probabilidade (`ai_decide`, no padrão System One visto na teoria) e aplicar um limiar de decisão.

## Parte 0 — O espaço, o grupo e o recorte (5 min)

A primeira célula reabre o espaço `blue-blood`. Na segunda, usem o mesmo recorte do lab 1. Se o lab 1 foi feito neste workspace, ele está em `gold.lab1_decisao`.

In [0]:
%sql
-- 🟢 PRONTA — cria (ou reabre) o espaço persistente da Blue Blood. Só execute.
-- Catálogo = onde ficam as tabelas. Zonas (schemas) = Bronze, Silver e Gold. Volume = onde ficam os arquivos.
-- Se reconectar o notebook, rode esta célula de novo: ela não apaga nada.
CREATE CATALOG IF NOT EXISTS `blue-blood` COMMENT 'Blue Blood (CAES-PMERJ / FGV): tabelas e arquivos dos labs';
USE CATALOG `blue-blood`;
CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Bronze: o dado como chegou, sem alteração';
CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Silver: limpo, tipado e checado';
CREATE SCHEMA IF NOT EXISTS gold COMMENT 'Gold: indicadores e as decisões e parâmetros do grupo';
CREATE VOLUME IF NOT EXISTS bronze.arquivos COMMENT 'Arquivos baixados como chegaram (ISPDados, narrativas)';
SHOW SCHEMAS

In [0]:
%sql
-- Parte 0 — o grupo e o recorte escolhido na aula 1.
CREATE OR REPLACE TABLE gold.lab2_recorte COMMENT 'Lab 2: grupo, CISP e delito acompanhados pelo grupo' AS
SELECT
  0 AS grupo,                -- TODO: número do grupo, de 1 a 8 (sem aspas)
  12 AS cisp,                -- TODO: número da CISP do grupo (sem aspas). 12 é só o exemplo da aula
  'roubo_rua' AS delito      -- TODO: 'roubo_rua', 'roubo_transeunte' ou 'roubo_celular'

---

# Lab 1 (manhã) — NLP em boletins

## Parte 1 — Carregar as narrativas na Bronze (10:00–10:15)

A célula Python baixa as 996 narrativas descaracterizadas do repositório público para o volume `bronze.arquivos`; a célula SQL seguinte transforma o arquivo em tabela. Sem internet, entra uma amostra de 12 narrativas do mesmo corpus. Nada aqui contém dado pessoal real: os nomes são fictícios.

In [0]:
%pip install openpyxl -q

In [0]:
# 🟢 PRONTA — baixa as narrativas para o volume da Bronze. Só execute.
# A planilha original fica no volume como chegou. Como o SQL não lê planilha, a célula grava ao lado
# uma cópia em JSON (mesmo conteúdo, só muda o formato), que a próxima célula lê em SQL.
# Sem internet, grava a amostra embutida de 12 narrativas do mesmo corpus.
import io, json, urllib.request
import pandas as pd

VOLUME = "/Volumes/blue-blood/bronze/arquivos"
URL_KDMILE = "https://github.com/murilogoes/bo-anonymizer-ptbr/raw/main/data/bos_anonimizacao_CAP_descaracterizado.xlsx"
AMOSTRA_EMBUTIDA = json.loads(r'''[
{
"Seq": 1,
"Texto Descaracterizado": "Pedro Paulo Da Silva foi preso em flagrante por roubo, após ter sido reconhecido pelas vítimas, Aloisio Freitas da Cunha e Patricia Hussein Brito, como o indivíduo que invadiu o estabelecimento comercial Prato Feliz, ameaçando-os com uma arma de fogo e subtraindo dinheiro, um celular da loja e o celular de Aloisio. A prisão foi convertida em preventiva, com base na gravidade do crime e no risco de reiteração delitiva.",
"Nome(s)": "Pedro Paulo Da Silva; Aloisio Freitas da Cunha; Patricia Hussein Brito; Aloisio",
"Localizações (s)": "Prato Feliz"
},
{
"Seq": 2,
"Texto Descaracterizado": "Presente nesta delegacia, Juliana Marta, informando que quando estava entrando em seu veículo que estava estacionado, foi abordada por um indivíduo dizendo que estava armado e que não era para ela gritar, senão iria atirar. O indivíduo levou uma corrente de ouro, 03 anéis e sua bolsa que continha seu celular, cartões de banco, RG e CNH. ",
"Nome(s)": "Juliana Marta",
"Localizações (s)": ""
},
{
"Seq": 3,
"Texto Descaracterizado": "Um cidadão foi abordado por dois indivíduos em uma moto Honda vermelha na Av. Dos Lagos, altura do número 367, no bairro Pedra Branca, por volta das 20:30h. O carona desceu da moto, anunciou o assalto e exigiu a mochila e os pertences da vítima, fugindo em seguida em direção desconhecida. O autor 1 permaneceu na moto e o autor 2, que desceu, era descrito como sem barba e sem óculos, vestindo camiseta do Corinthians. A placa da moto não foi identificada, mas a mesma estava amassada.",
"Nome(s)": "",
"Localizações (s)": "Av. Dos Lagos; Pedra Branca"
},
{
"Seq": 4,
"Texto Descaracterizado": "A vítima relatou que foi abordada por dois criminosos que a ameaçaram e a imobilizaram com uma \"gravata\", roubando seus pertences. A vítima não conseguiu identificar os autores do crime.",
"Nome(s)": "",
"Localizações (s)": ""
},
{
"Seq": 5,
"Texto Descaracterizado": "Comparecem os policiais militares Abreu e Vera, integrantes da VTR A12345, noticiando que durante deslocamento para atendimento de ocorrência,  avistaram uma motocicleta estacionada e com dois indivíduos nela embarcados. Em seguida, outra motocicleta, de modelo indeterminado, sendo pequeno porte e cor escura, com dois indivíduos, parou ao lado da primeira, um deles, desceu e foi ao encontro dos outros dois embarcados. A guarnição adentrou a via. Assim que os indivíduos avistaram a viatura, o que estava embarcado na moto de pequeno porte, fugiu na direção da Rua do Tanque. Os que estavam na Honda PCX, cor marrom, abandonaram a motocicleta no chão e empreenderam fuga, sendo que dois deles seguiram pelo telhado de uma casa vizinha e um deles foi alcançado. Próximo à Honda PCX foi localizado um boné e um aparelho de celular. Ao que indica, pertence a um dos indivíduos que fugiu. O abordado é o adolescente José Alfredo Pinto. Nada de ilícito foi encontrado, porém em busca nos sistemas policiais, a motocicleta é produto de roubo ocorrido nesta data às 18:30, nas proximidades do Shopping Botelho, na Avenida Ribeiro. Os PMs estavam com as câmeras ligadas, porém não acionadas. Foi utilizado algemas devido ao receio de fuga. A mãe do adolescente compareceu no local dos fatos. O adolescente, a motocicleta, o boné e o celular localizados foram conduzidos à esta delegacia e colocado à disposição da autoridade policial.  Presente nesta unidade, a vítima Maria Pereira da Silva nos relatou que estava a caminho do trabalho, por volta das 18:30, pela Travessa Sem Bar, altura do numeral 50, no município de Itaquaquecetuba, quando foi abordada por dois indivíduos numa moto, modelo não identificado. Eles interceptaram a sua moto após a lombada, ameaçaram com arma e ordenaram que ela descesse da sua motocicleta.  Não levaram capacete e nem celular, mas levaram o que estava no interior do baú. Ela seguiu até o posto de combustível próximo do local e acionou 190 para comunicar os fatos. Vítima compartilhou a notícia do roubo de sua moto nas redes sociais e foi avisada por uma amiga que viu uma moto parecida com a sua numa abordagem policial. Ela se dirigiu ao local e confirmou ser a sua motocicleta, sendo instruída pelos policiais a se dirigir a esta delegacia.  Na delegacia, foi realizado o Reconhecimento de Pessoa, mas a vítima não reconheceu o adolescente como sendo um dos autores do Roubo de sua motocicleta.  Indagado a respeito dos fatos, na presença de sua genitora, ora curadora, o adolescente  José Alfredo Pinto declarou que estava em casa quando decidiu ir visitar o amigo de nome Tubarão. No caminho, viu que uma viatura da PM estava abordando uma moto e os moleques jogaram esta moto no chão e saíram correndo. José se assustou, correu viu o portão aberto e entrou no quintal da casa de uma mulher. Na fuga, viu um outro moleque pulando o muro desta mesma casa. Os policiais entraram e pegaram José no quintal. Ordenaram que ele deitasse no chão, revistaram e perguntaram sobre a moto, mas ele desconhece a moto e os moleques que ali estavam. Informou, ainda que não faz uso de álcool e nem de entorpecentes. Ele não está estudando porque perdeu a vaga por excesso de faltas. A mãe informa que entrou com pedido de nova vaga, mas enquanto isso ele cuida dos irmãos para ela trabalhar. Seu genitor é ausente. Mãe informa que é a primeira vez que comparece em delegacia e ele nunca levou nada de diferente para dentro de casa. O que ele tem, ela comprou.  Ciente de tais fatos, após entrevista reservada com os envolvidos, nesta etapa urgente de cognição sumaríssima, este signatário formou sua decisão e convicção jurídica, entendendo não presentes elementos para se atribuir ao adolescente José a prática do roubo do veículo da vítima, restando imputação pelo ato infracional análogo ao delito prescrito no artigo 180 caput do Código Penal.  Não se tratando de ato infracional  praticado com violência ou grave ameaça, determino a liberação do adolescente infrator ao seu responsável legal, mediante a assinatura do termo de compromisso e responsabilidade, conforme preceitua o artigo 174 do Estatuto da Criança e do Adolescente, bem como a elaboração do presente boletim de ocorrência circunstanciado e seu posterior encaminhamento ao Ministério Público. ",
"Nome(s)": "Abreu; Vera; José Alfredo Pinto; Maria Pereira da Silva",
"Localizações (s)": "Rua do Tanque; Shopping Botelho; Avenida Ribeiro; Travessa Sem Bar"
},
{
"Seq": 6,
"Texto Descaracterizado": "Descrição ocorrência cidadão: Me acordou próximo ao Mercadinho do João aonde eu desci do ônibus e precisava me deslocar até o outro ponto  , me pediu dinheiro , informei que não tinha  , começou a ficar apreensivo.  Agressivo em palavras , começou a tentar puxa minha mochila.  Puxou e saiu correndo ,  arranhou meu braço  Histórico do boletim: A redação do presente Boletim Eletrônico de Ocorrência foi elaborada pelo Declarante e as informações nele constantes são de sua exclusiva responsabilidade.     Este B.E.O. não substitui os documentos originais.    IMPORTANTE:   - Caso RG emitido no Estado de São Paulo tenha sido roubado e listado neste B.E.O., ficará bloqueado em caráter irreversível e nova via deverá ser providenciada;   - Mesmo que os documentos sejam recuperados, não haverá possibilidade de cancelamento deste B.E.O..  BO digitado por Maria Pereira Silva, equipe Prodesp.   - Autor 1 – Desconhecido, Barba: Barba cheia ",
"Nome(s)": "Maria Pereira Silva",
"Localizações (s)": "Mercadinho do João"
},
{
"Seq": 7,
"Texto Descaracterizado": "Descrição ocorrência cidadão: Estava Descendo a Rua para chegar em casa , fui abordado por dois indivídos em uma moto de características preta os dois estavam de moletom preto capacete preto um deles com detalhe vermelho no capacete , apontaram a arma na minha cabeça e pegaram o celular pediram carteira mas não tinha somente o celular , a última localização do aparelho estava constando entre a Rua do Tanque, Avenida Ribeiro, 2000 Histórico do boletim: A redação do presente Boletim Eletrônico de Ocorrência foi elaborada pelo Declarante e as informações nele constantes são de sua exclusiva responsabilidade.     IMPORTANTE:   Mesmo que o aparelho seja recuperado, não haverá possibilidade de cancelamento deste B.E.O..    B.O Digitado por João Andrade de Souza, Equipe Prodesp. - Autor 1 – Desconhecido, Barba: Sem barba  - Autor 2 – Desconhecido, Barba: Sem barba   VEÍCULO: Não Especificado Pessoa Relacionada: Autor 1 – Desconhecido Placa: Não informado Tipo de Veículo: Moto Marca/Modelo: Não informado Cores: Preta Estado (UF): SP Cidade: Não informado Proprietário do Veículo: Não informado Características adicionais do veículo úteis para a investigação: Moto preta aparentava ser uma 160 Com a placa meio dobrada . ",
"Nome(s)": "João Andrade de Souza",
"Localizações (s)": "Rua do Tanque; Avenida Ribeiro, 2000"
},
{
"Seq": 8,
"Texto Descaracterizado": "Um cidadão foi abordado por dois indivíduos em uma moto preta, na Rua do Tanque, enquanto caminhava para casa. Os indivíduos, que usavam moletom preto e capacetes pretos (um com detalhe vermelho), apontaram uma arma para a vítima e roubaram seu celular. A última localização do aparelho foi registrada entre a Rua do Tanque e a Avenida Ribeiro.",
"Nome(s)": "",
"Localizações (s)": "Rua do Tanque; Avenida Ribeiro"
},
{
"Seq": 10,
"Texto Descaracterizado": "Descrição ocorrência cidadão: 5 ou 4 caras montado em 3 motos vieram pelo lado do parceiro e anunciou o assalto só tinha um armado com um 38 mais ou menos só lembro que era uma arma com os negócios que gira as balas uma moto era branca ???  ... ???     ... CONSIDERAÇÕES DO DELEGADO DE POLÍCIA: a narrativa foi realizada pela própria vítima, utilizando-se da plataforma do Boletim de Ocorrência Eletrônico e, por ora, com a análise dos dados firmados pelo usuário dos serviços, conclui-se que o caso tem feições do crime acima especificado. .  Caso a vítima tenha conhecimento de informações que possam elucidar a determinação de autoria, fica desde já orientada a comparecer no Setor de Investigações da Delegacia de Polícia do local dos fatos, juntamente com o presente Boletim de Ocorrência, para compartilhamento de informações.   Registro a ser levado para deliberação do Exmo. Sr. Dr. Delegado de Polícia titular da unidade do local dos fatos. Nada mais.  ",
"Nome(s)": "",
"Localizações (s)": ""
},
{
"Seq": 11,
"Texto Descaracterizado": "Comparecem nesta Delegacia de Polícia os policiais militares acima qualificados narrando os seguintes fatos - que na data de 14/04/2005 quatro indivíduos tentaram roubar o estabelecimento comercial “Farmácia do Souza” situado no endereço, Rua do Tanque, numeral 100, bairro Poiares. O gerente do citado estabelecimento ao perceber a atitude suspeita dos indivíduos, solicitou para que um outro funcionário, acionasse o alarme do local. Diante do fato, os indivíduos se evadiram do interior do estabelecimento, e ainda tentaram roubar um cliente que estava no estacionamento. Porém não obtiveram êxito.  Seguiram então sentido região sul desta cidade, momento em que abordaram a pessoa de João Andrade de Souza que naquele momento estava se dirigindo para o trabalho, e fazendo menção de que estavam armados, solicitaram o celular dele. Via COPOM, a equipe policial foi notificada sobre a tentativa de roubo na “Farmácia do Souza”. Prontamente se dirigiram ao local onde o gerente narrou o ocorrido acima, e informou que os indivíduos teriam fugido no sentido sul desta cidade. Ato contínuo em busca dos indivíduos, a equipe policial avistou um indivíduo, no caso a vítima, solicitando auxílio. Vítima os alertou de que havia sido roubado, e apontou a direção que os indivíduos tomaram rumo. Em abordagem pessoal, nada de ilícito foi encontrado com os indivíduos, porém um deles, José Alfredo Pinto, estava com o objeto utilizado para efetuar o roubo do aparelho celular, no caso uma tesoura. O aparelho celular no momento estava em posse do indiciado.  Ante ao exposto, a equipe policial conduziu a vítima e os quatro indivíduos até este distrito policial para a apreciação da autoridade policial plantonista. Foi necessário o uso de algemas no indiciado e nos adolescentes infratores tanto para a segurança deles quanto da equipe policial. Os indivíduos não se encontram lesionados. Expediu-se requisição de exame de corpo de delito cautelar, visando atestar a integridade física dos indivíduos, sendo determinada a fotografação deles de corpo inteiro.  I - DAS OITIVAS 1. Depoimento da Vítima A vítima João Andrade de Souza declarou que foi abordada pouco antes das 3h da madrugada quando se dirigia ao trabalho de bicicleta. Relatou que quatro indivíduos em bicicletas o imprensaram contra um ponto de ônibus, fizeram menção de estarem com arma de fogo escondida sob uma blusa de frio e apontaram uma tesoura para si. Os indivíduos o revistaram e subtraíram seu aparelho celular, ordenando que não olhasse para trás ao sair. Após se afastar, encontrou uma viatura policial, relatou o ocorrido e, posteriormente, reconheceu o celular recuperado pelos policiais como sendo de sua propriedade. A vítima afirmou com convicção reconhecer os quatro indivíduos apresentados como os autores do roubo.  2. Depoimento dos Policiais Militares Condutores O PM Abreu relatou que a equipe policial foi acionada via rede rádio para averiguar uma tentativa de roubo à Farmácia do Souza. Durante o deslocamento, foram solicitados por um transeunte (vítima João) que informou ter sido roubado por quatro indivíduos, com características compatíveis com os autores da tentativa de roubo à farmácia. A vítima informou a direção de fuga dos suspeitos, e a equipe policial os localizou embaixo da ponte da Besta, no bairro Porto Novo . Na abordagem, foi encontrado o celular roubado com o indivíduo maior de idade. O policial confirmou o uso de algemas, mas negou resistência na abordagem.  O PM Costa corroborou o depoimento do seu parceiro, acrescentando que durante a revista pessoal, o aparelho celular foi localizado com o maior de idade, e no último indivíduo (José Alfredo Pinto) foi encontrada uma tesoura. Informou também que a vítima relatou que os autores simularam estar armados. Confirmou o uso de algemas justificado pelo receio de fuga e pela superioridade numérica dos abordados.  3. Interrogatório do Indiciado Guilherme Pereira, 18 anos, ao ser interrogado, exerceu seu direito constitucional de permanecer em silêncio. Ao final do interrogatório, quando questionado sobre as condições da abordagem policial, alegou ter recebido \"um soco na cabeça\" e que as algemas foram colocadas com muita força.  4. Depoimento dos Adolescentes Infratores Marcos dos Santos, 15 anos, optou por permanecer em silêncio, exercendo seu direito constitucional de não produzir provas contra si mesmo.  Pedro Minotauro, 15 anos, inicialmente optou por prestar declarações, confirmando que esteve na farmácia com os demais, todos encapuzados. Alegou que não fizeram nada na farmácia e que teriam ido embora. Quando questionado sobre o encontro com a vítima, apresentou contradições em suas declarações, inicialmente confirmando e depois negando o envolvimento no roubo do celular.  José Alfredo Pinto, 16 anos, optou por permanecer em silêncio, exercendo seu direito constitucional.  II - DA MATERIALIDADE E AUTORIA A materialidade delitiva encontra-se demonstrada pelo Auto de Exibição e Apreensão que documenta a apreensão do aparelho celular da vítima em poder do indiciado Guilherme Pereira, bem como da tesoura utilizada como instrumento para intimidação durante o roubo, encontrada com o adolescente José Alfredo Pinto. Consta também a Requisição de Exame Pericial dos objetos apreendidos.  III - DA TIPIFICAÇÃO Diante do conjunto probatório colhido, verifico que a conduta do indiciado Guilherme Pereira se amolda, em tese, aos tipos penais previstos no artigo 157, §2º, inciso II do Código Penal (roubo qualificado pelo concurso de pessoas) e artigo 244-B do Estatuto da Criança e do Adolescente (corrupção de menores), em concurso material (art. 69 do CP).  Quanto aos adolescentes José Alfredo Pinto, Marcos dos Santos e Pedro Minotauro, suas condutas caracterizam, em tese, ato infracional análogo ao crime previsto no artigo 157, §2º, inciso II do Código Penal.  iV - DA HOMOLOGAÇÃO DO FLAGRANTE Considerando que a prisão em flagrante atende aos requisitos legais estabelecidos nos artigos 301 e 302, incisos I e II do Código de Processo Penal, não apresentando quaisquer vícios formais ou materiais, HOMOLOGO o presente auto de prisão em flagrante.  V - DELIBERAÇÕES Ante o exposto, DETERMINO:  A LAVRATURA DO AUTO DE PRISÃO EM FLAGRANTE do indiciado Guilherme Pereira, nos termos do Código de Processo Penal;  A LAVRATURA DO AUTO DE APREENSÃO dos adolescentes José Alfredo Pinto, Marcos dos Santos e Pedro Minotauro, nos termos do Estatuto da Criança e do Adolescente;  A expedição de nota de culpa ao autuado;  A expedição de ofício à autoridade judiciária competente, comunicando a prisão;  A expedição de ofício ao Ministério Público, para conhecimento;  A instauração do Inquérito Policial para apuração completa dos fatos;  A remessa dos adolescentes infratores ao Juízo da Infância e Juventude;  O encaminhamento do indiciado maior de idade ao sistema prisional.",
"Nome(s)": "João Andrade de Souza; José Alfredo Pinto; João; Guilherme Pereira; Marcos dos Santos; Pedro Minotauro; Abreu; Costa",
"Localizações (s)": "Farmácia do Souza; Rua do Tanque, numeral 100; ponte da Besta; bairro Poiares; bairro Porto Novo"
},
{
"Seq": 12,
"Texto Descaracterizado": "Neste dia e horário, na sede da Central de Polícia Judiciária de Praia Grande, comparecem os Policiais Militares acima identificados narrando que estavam em patrulhamento de rotina, quando foram informados pela VTR A12345, encarregado CB PM Abreu, que um indivíduo havia subtraído pertences e dinheiro das vítimas do estabelecimento comercial de nome Restaurante Coma Bem, situado Rua do Tanque, nº. 100, bairro Boqueirão, indicando as características do autor. Informou ainda que o rastreio do celular da vítima estava no bairro Sítio do Campo, em deslocamento pelas ruas daquela localidade. Em patrulhamento com vistas, o indivíduo foi localizado de bicicleta e abordado na Avenida Ribeiro, altura do nº. 2.000, bairro Sítio do Campo, identificado como João Andrade de Souza. Segundos antes da abordagem, avistaram o autor dispensando ao chão algo, o qual foi recuperado e se tratava de um objeto metálico simulando uma arma de fogo. Após busca pessoal, na mão do autor foram localizados dois celulares e moedas, envoltos em uma blusa. Diante dos fatos, foi conduzido a esta Delegacia de Polícia para as providências cabíveis, sendo necessário o uso de algemas, diante de fundado receio de fuga. A bicicleta foi apreendida administrativamente pela Polícia Militar.  A ofendida Maria Pereira Silva afirma que trabalha no estabelecimento comercial Restaurante Coma Bem, na função de auxiliar de cozinha, e estava limpando a cozinha para fechar e José estava na frente recebendo os pedidos. Em dado momento, um cidadão adentrou na cozinha, com uma arma nas costas do José, exigindo que a declarante entregasse dinheiro e seu celular. Havia um celular carregando no balcão da cozinha e o autor o puxou, apoderando-se do bem, que é de propriedade de José. A declarante disse que era funcionária e não tinha dinheiro. O indivíduo voltou para a parte da frente e começou a revirar o caixa. Ao fim, o indivíduo se evadiu, tomando rumo ignorado. A genitora de José desceu as escadas e acionou a Polícia. Atesta que não teve prejuízo financeiro. Após reconhecimento pessoal, reconhece o agora identificado João Andrade de Souza como autor do delito, com 100% de certeza.   O ofendido José Alfredo Pinto, na presença de sua genitora, afirma que possui quatorze anos de idade e deseja revelar espontaneamente o ocorrido a este Delegado de Polícia. No horário dos fatos, estava na parte da frente do estabelecimento comercial Restaurante Coma Bem, recebendo os pedidos online de clientes. A porta da frente estava trancada, pois é delivery. Em dado momento, um homem tentou abrir a porta e perguntou se a loja estava aberta. O declarante disse que sim e foi lá abrir, pois pensou que se tratava de um cliente. O indivíduo entrou e, quanto o declarante se virou para o balcão, foi abordado com um objeto parecido com arma, que disse: “se afasta; se afasta”, gesticulando, então o declarante entrou na cozinha. O homem passou a revirar o caixa e se apoderou das notas e moedas. O indivíduo visualizou um celular que estava carregando em cima da bancada e se apoderou, puxando, celular este que é da loja. Pediu que o declarante saísse da cozinha e o confrontou se o que tinha no caixa era tudo o que tinha no comércio. O declarante afirmou que sim e o autor olhou novamente o balcão e se apoderou do seu celular que estava em cima do balcão do caixa, ao lado do monitor. Ao fim, o indivíduo se evadiu. Começou a gritar por sua mãe que estava na parte de cima do estabelecimento, a qual desceu e se inteirou do ocorrido, pois nada presenciou. Sua genitora acionou a Polícia Militar, passando a localização do telefone. Após reconhecimento pessoal, reconhece o agora identificado João Andrade de Souza como autor do delito, com 100% de certeza. Seu celular se trata de um Samsung Galaxy M55, avaliado em R$ 1.500,00 (mil e quinhentos reais); o da loja se trata de um Multilaser F Pro 2, avaliado em R$ 400,00 (quatrocentos reais), além da quantia de R$ 58,31 (cinqüenta e oito reais e trinta e um centavos) em dinheiro do caixa.  João Andrade de Souza, antes de ser ouvido por esta Autoridade Policial foi cientificado de seus direitos constitucionais, especialmente o de permanecer em silêncio durante o interrogatório policial e falar somente em juízo. Declara que irá se manifestar apenas em juízo. Não indicou o número de ninguém para ser informado de sua prisão  Examinadas as versões e demais elementos amealhados, o Delegado de Policia exarou sua decisão e convicção jurídica (Lei Federal n°12.830/2013, art.2°, parágrafo 6°): os elementos apresentados, em especial a firme narrativa dos ofendidos e o reconhecimento pessoal, bem como a forma que se deu a abordagem, com o autor ainda em posse da res furtiva, são suficientes para apontar a prática do crime tipificado no artigo 157, caput, do Código Penal.  Por tal razão, foi DECRETADA, neste instante, a PRISÃO EM FLAGRANTE de João Andrade de Souza, determinando-se o prosseguimento do procedimento nos seus ulteriores termos.  A Autoridade Policial, na etapa de cognição sumaríssima, reputou configurado o estado flagrancial, já que o autor tinha acabado de cometer a infração penal (CPP, art. 302, II).   Considerando a pena máxima cominada em abstrato suplanta quatro anos, deixo de arbitrar fiança, permanecendo o indiciado custodiado. Determino ainda o FORMAL INDICIAMENTO do agente, com entrega da correspondente nota de culpa.   Autorizo a entrega da res furtiva ao legítimo proprietário, presentes a este plantão policial. Determino ainda que os vídeos das câmeras de segurança do comércio sejam disponibilizados via Portal de Vídeos. Determino a apreensão do objeto utilizado como “arma” para ameaçar as vítimas.   Comunique-se a Autoridade Judicial competente, com promoção das demais comunicações e cerimônias legais pertinentes. Juntem-se, ademais, oitivas e documentação pertinente, integrantes deste. Instaure-se Inquérito Policial.   DA REPRESENTAÇÃO PARA CONVERSÃO DA PRISÃO EM FLAGRANTE EM PRISÃO PREVENTIVA  Conforme previsão do art. 282, incisos I e II do CPP, a aplicação das medidas cautelares se dá de acordo com os Princípios da Necessidade e da Adequação. No caso em comento, a segregação cautelar se mostra como a medida mais necessária e adequada.  Diante da gravidade concreta do crime, bem como das circunstâncias do autor e do risco de ineficácia de outras medidas cautelares diversas da prisão, REPRESENTA-SE pela CONVERSÃO DA PRISÃO EM FLAGRANTE EM PRISÃO PREVENTIVA do indiciado João Andrade de Souza, nos termos dos arts. 282 § 6º, 311, 312, e 313 inc. I, todos do Código de Processo Penal, a fim de garantir da ordem pública e para assegurar a aplicação da lei penal. ",
"Nome(s)": "Abreu; João Andrade de Souza; Maria Pereira Silva; José; José Alfredo Pinto",
"Localizações (s)": "Restaurante Coma Bem; Rua do Tanque, nº. 100; bairro Boqueirão; Avenida Ribeiro, altura do nº. 2.000; bairro Sítio do Campo"
},
{
"Seq": 13,
"Texto Descaracterizado": "O declarante, após pegar o trem errado, pediu informações a um morador de rua, que lhe roubou o celular. Outro morador interveio, recuperando o aparelho, mas a capinha e os documentos que estavam dentro foram perdidos. O Boletim de Ocorrência foi registrado e o RG do declarante, caso tenha sido emitido em São Paulo, ficará bloqueado.",
"Nome(s)": "",
"Localizações (s)": ""
}
]''')
COLUNAS = {"Seq": "seq", "Texto Descaracterizado": "texto", "Nome(s)": "nomes", "Localizações (s)": "localizacoes",
           "Documento(s)": "documentos", "Emplacamento (s)": "placas", "Telefone (s)": "telefones"}

try:
    with urllib.request.urlopen(URL_KDMILE, timeout=60) as r:
        planilha = r.read()
    with open(f"{VOLUME}/bos_anonimizacao_CAP_descaracterizado.xlsx", "wb") as f:
        f.write(planilha)
    narr = pd.read_excel(io.BytesIO(planilha))
    ORIGEM_NARRATIVAS = "KDMiLe 2026, repositório público (CC BY 4.0)"
except Exception as erro:
    narr = pd.DataFrame(AMOSTRA_EMBUTIDA)
    ORIGEM_NARRATIVAS = f"Amostra embutida (12 narrativas do KDMiLe); download falhou: {type(erro).__name__}"

narr = narr.rename(columns=COLUNAS).reindex(columns=list(COLUNAS.values()))
with open(f"{VOLUME}/narrativas_kdmile.json", "w", encoding="utf-8") as f:
    for linha in narr.to_dict("records"):
        linha = {k: (None if pd.isna(v) or str(v).strip() == "" else (int(v) if k == "seq" else str(v)))
                 for k, v in linha.items()}
        f.write(json.dumps(linha, ensure_ascii=False) + "\n")
spark.sql("""CREATE TABLE IF NOT EXISTS bronze.registro_ingestao
             (arquivo STRING, origem STRING, url STRING, bytes BIGINT, baixado_em TIMESTAMP)
             COMMENT 'Bronze: de onde veio cada arquivo e quando'""")
spark.sql("INSERT INTO bronze.registro_ingestao VALUES ('narrativas_kdmile.json', :origem, :url, :n, current_timestamp())",
          args={"origem": ORIGEM_NARRATIVAS, "url": URL_KDMILE, "n": len(narr)})
print(f"{ORIGEM_NARRATIVAS}: {len(narr)} narrativas gravadas em {VOLUME}")

In [0]:
%sql
-- 🟢 PRONTA — Bronze: as narrativas viram tabela, sem alteração. Só execute.
CREATE OR REPLACE TABLE bronze.narrativas_kdmile
COMMENT 'Bronze: narrativas de roubo descaracterizadas (KDMiLe 2026, CC BY 4.0) com o padrão-ouro de dados pessoais'
AS SELECT CAST(seq AS INT) AS seq, texto, nomes, localizacoes, documentos, placas, telefones,
          current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/blue-blood/bronze/arquivos/narrativas_kdmile.json', format => 'json',
                schema => 'seq BIGINT, texto STRING, nomes STRING, localizacoes STRING, documentos STRING, placas STRING, telefones STRING');

SELECT count(*) AS narrativas, substr(max_by(texto, seq), 1, 300) AS exemplo FROM bronze.narrativas_kdmile

## Parte 2 — Classificar por regras e auditar (10:15–10:45)

**Atividade 2.1 (10 min).** O dicionário de palavras é uma **tabela de parâmetros** (`gold.lab2_dicionario`): cada linha diz que uma palavra marca um rótulo de **o que foi levado** ou do **meio empregado**. É o jeito mais simples de NLP: transparente e fácil de corrigir. O grupo pode acrescentar linhas (opcional) e rodar as duas células de novo.

In [0]:
%sql
-- Atividade 2.1 — dicionário de regras. Opcional: acrescente linhas (palavra sem acento, em minúsculas).
CREATE OR REPLACE TABLE gold.lab2_dicionario COMMENT 'Lab 2: regras de NLP, palavra que marca cada rótulo' AS
SELECT * FROM VALUES
  ('objeto', 'Celular', 'celular'), ('objeto', 'Celular', 'telefone movel'), ('objeto', 'Celular', 'smartphone'),
  ('objeto', 'Celular', 'iphone'),
  ('objeto', 'Veículo (carro ou moto)', 'veiculo'), ('objeto', 'Veículo (carro ou moto)', 'carro'),
  ('objeto', 'Veículo (carro ou moto)', 'automovel'), ('objeto', 'Veículo (carro ou moto)', 'moto'),
  ('objeto', 'Estabelecimento comercial', 'loja'), ('objeto', 'Estabelecimento comercial', 'estabelecimento'),
  ('objeto', 'Estabelecimento comercial', 'comercio'), ('objeto', 'Estabelecimento comercial', 'mercado'),
  ('objeto', 'Estabelecimento comercial', 'farmacia'), ('objeto', 'Estabelecimento comercial', 'padaria'),
  ('objeto', 'Residência', 'residencia'), ('objeto', 'Residência', 'casa da vitima'), ('objeto', 'Residência', 'apartamento'),
  ('objeto', 'Transporte coletivo', 'onibus'), ('objeto', 'Transporte coletivo', 'coletivo'),
  ('objeto', 'Bicicleta', 'bicicleta'),
  ('meio', 'Arma de fogo', 'arma de fogo'), ('meio', 'Arma de fogo', 'revolver'), ('meio', 'Arma de fogo', 'pistola'),
  ('meio', 'Arma de fogo', 'armado'),
  ('meio', 'Arma branca', 'faca'), ('meio', 'Arma branca', 'canivete'), ('meio', 'Arma branca', 'arma branca'),
  ('meio', 'Arma branca', 'facao'),
  ('meio', 'Simulacro', 'simulacro'), ('meio', 'Simulacro', 'arma de brinquedo')
AS t(tipo, rotulo, palavra)

In [0]:
%sql
-- 🟢 PRONTA — aplica o dicionário (Silver) e conta (Gold). Só execute.
CREATE OR REPLACE TABLE silver.narrativas_classificadas
COMMENT 'Silver: narrativas com o que foi levado e o meio empregado, pelas regras de gold.lab2_dicionario' AS
WITH n AS (
  SELECT seq, texto, translate(lower(texto), 'áàâãäéèêëíìîïóòôõöúùûüç', 'aaaaaeeeeiiiiooooouuuuc') AS t
  FROM bronze.narrativas_kdmile
)
SELECT n.seq, n.texto,
       array_sort(collect_set(CASE WHEN d.tipo = 'objeto' THEN d.rotulo END)) AS objeto,
       coalesce(min(CASE WHEN d.tipo = 'meio' THEN d.rotulo END), 'Sem arma descrita') AS meio
FROM n LEFT JOIN gold.lab2_dicionario d ON instr(n.t, d.palavra) > 0
GROUP BY n.seq, n.texto;

CREATE OR REPLACE TABLE gold.narrativas_por_rotulo
COMMENT 'Gold: quantas narrativas têm cada rótulo de objeto e de meio' AS
SELECT 'O que foi levado' AS pergunta, coalesce(o, 'Nenhum objeto identificado') AS rotulo, count(*) AS narrativas
FROM silver.narrativas_classificadas LATERAL VIEW OUTER explode(objeto) x AS o
GROUP BY coalesce(o, 'Nenhum objeto identificado')
UNION ALL
SELECT 'Meio empregado', meio, count(*) FROM silver.narrativas_classificadas GROUP BY meio;

SELECT * FROM gold.narrativas_por_rotulo ORDER BY pergunta DESC, narrativas DESC

**Atividade 2.2 (15 min) — auditoria.** Toda classificação automática precisa ser conferida. A célula sorteia 10 narrativas (o sorteio muda com o número do grupo) e guarda o sorteio em `gold.lab2_amostra_auditoria`. Leiam cada uma e digam se as marcas estão **certas**, **erradas** ou **incompletas**.

In [0]:
%sql
-- 🟢 PRONTA — sorteia 10 narrativas para auditar. Só execute.
CREATE OR REPLACE TABLE gold.lab2_amostra_auditoria COMMENT 'Lab 2: as 10 narrativas sorteadas para a auditoria do grupo' AS
SELECT * FROM (
  SELECT row_number() OVER (ORDER BY xxhash64(c.seq, r.grupo)) AS item, c.seq,
         array_join(c.objeto, ', ') AS objeto, c.meio,
         concat(substr(c.texto, 1, 350), CASE WHEN length(c.texto) > 350 THEN '...' ELSE '' END) AS trecho
  FROM silver.narrativas_classificadas c CROSS JOIN gold.lab2_recorte r
) WHERE item <= 10;

SELECT * FROM gold.lab2_amostra_auditoria ORDER BY item

In [0]:
%sql
-- Atividade 2.2 — uma avaliação por item sorteado: 'certo', 'errado' ou 'incompleto'.
CREATE OR REPLACE TABLE gold.lab2_auditoria COMMENT 'Lab 2: auditoria humana da classificação por regras' AS
SELECT * FROM VALUES
  (1, 'TODO'), (2, 'TODO'), (3, 'TODO'), (4, 'TODO'), (5, 'TODO'),
  (6, 'TODO'), (7, 'TODO'), (8, 'TODO'), (9, 'TODO'), (10, 'TODO')
AS t(item, avaliacao);

SELECT CASE WHEN count_if(avaliacao IN ('certo', 'errado', 'incompleto')) = 0 THEN 'Preencha a auditoria e rode de novo.'
            ELSE concat('Acerto das regras na auditoria: ', count_if(avaliacao = 'certo'), '/',
                        count_if(avaliacao IN ('certo', 'errado', 'incompleto')), ' (',
                        CAST(round(100 * count_if(avaliacao = 'certo') / count_if(avaliacao IN ('certo', 'errado', 'incompleto'))) AS INT),
                        '%). Para comparar: modelos BERT chegaram a 89%–90% de acurácia em 10 classes de crime (Alves et al., 2024).')
       END AS resultado
FROM gold.lab2_auditoria

## Parte 3 — O que a máquina encontra sozinha (10:45–11:05)

Agora sem regras: a célula transforma cada narrativa em números (TF-IDF, o peso de cada palavra) e junta as parecidas em 6 grupos (k-means). Ninguém disse à máquina o que procurar. O trabalho do gestor é **dar nome** a cada grupo e dizer se ele serve para alguma decisão.

> Esta é uma das poucas células em Python: o agrupamento (k-means) não existe em SQL. Ela lê a Silver e grava o resultado em `silver.narrativas_grupo_texto` e `gold.grupos_de_texto`.

In [0]:
# 🟢 PRONTA — agrupa as narrativas em até 6 grupos e mostra as palavras de cada um. Só execute.
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

PALAVRAS_VAZIAS = """a o e é de da do das dos em no na nos nas um uma uns umas que se por para com sem ao aos à às
pelo pela pelos pelas seu sua seus suas ele ela eles elas foi era estava ser ter tinha como mais mas ou já não sim lhe
lhes quando onde após sobre entre até também isso este esta esse essa vítima vitima indivíduo individuo indivíduos
individuos declarante relata informa informou momento local dia data hora horas cerca""".split()

narr = spark.table("silver.narrativas_classificadas").select("seq", "texto").orderBy("seq").toPandas()
k = min(6, max(2, len(narr) // 3))
vetor = TfidfVectorizer(stop_words=PALAVRAS_VAZIAS, max_df=0.5, min_df=2 if len(narr) < 100 else 5,
                        token_pattern=r"(?u)\b[^\W\d_]{3,}\b")
X = vetor.fit_transform(narr["texto"].fillna(""))
modelo = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
termos = np.array(vetor.get_feature_names_out())
resumo = pd.DataFrame({
    "grupo": range(k),
    "narrativas": [int((modelo.labels_ == g).sum()) for g in range(k)],
    "palavras_mais_marcantes": [", ".join(termos[np.argsort(modelo.cluster_centers_[g])[::-1][:8]]) for g in range(k)],
})
(spark.createDataFrame(pd.DataFrame({"seq": narr["seq"], "grupo_texto": modelo.labels_.astype(int)}))
 .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.narrativas_grupo_texto"))
spark.createDataFrame(resumo).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.grupos_de_texto")
display(spark.table("gold.grupos_de_texto").orderBy("grupo"))

In [0]:
%sql
-- Atividade 3.1 — deem um nome a cada grupo (0 a 5) e digam qual ajuda uma decisão do comando.
CREATE OR REPLACE TABLE gold.lab2_nomes_grupos COMMENT 'Lab 2: o nome que o grupo deu a cada grupo de texto' AS
SELECT * FROM VALUES
  (0, 'TODO'),   -- ex.: 'roubo por dupla em moto'
  (1, 'TODO'),   -- ex.: 'registro eletrônico'
  (2, 'TODO'),
  (3, 'TODO'),
  (4, 'TODO'),   -- se a célula acima mostrou menos de 6 grupos, deixe as linhas que sobram como estão
  (5, 'TODO')
AS t(grupo, nome);

CREATE OR REPLACE TABLE gold.lab2_grupo_mais_util COMMENT 'Lab 2: o grupo de texto que ajuda uma decisão, e qual' AS
SELECT 'TODO' AS grupo_e_decisao;   -- TODO: qual grupo ajuda uma decisão do comando, e qual decisão?

SELECT g.grupo, n.nome, g.narrativas, g.palavras_mais_marcantes
FROM gold.grupos_de_texto g LEFT JOIN gold.lab2_nomes_grupos n ON n.grupo = g.grupo ORDER BY g.grupo

## Parte 4 — Anonimizar e medir o que vaza (11:05–11:35)

O histórico do BO não é público porque tem nomes, locais, placas e documentos (LAI, art. 31; LGPD). Para compartilhar texto, é preciso anonimizar. A célula SQL aplica dois níveis de um anonimizador simples (expressões regulares) e mede, contra o padrão-ouro do corpus, **quanto de cada tipo de dado foi removido** (recall de segurança).

- **Nível 1:** troca placas, telefones, números de documento e sequências de duas ou mais palavras com inicial maiúscula.
- **Nível 2:** também troca palavras isoladas com inicial maiúscula fora do início de frase. Remove mais, mas apaga mais texto útil.

**Atividade 4.1.** A Corregedora escolhe o nível a examinar e o grupo decide com quem o texto anonimizado pode ser compartilhado. As narrativas mascaradas ficam em `silver.narrativas_anonimizadas`.

In [0]:
%sql
-- Atividade 4.1 — escolha o nível de anonimização a examinar.
CREATE OR REPLACE TABLE gold.lab2_anonimizacao_teste COMMENT 'Lab 2: nível do anonimizador examinado pelo grupo' AS
SELECT 1 AS nivel   -- TODO: 1 ou 2

In [0]:
%sql
-- 🟢 PRONTA — anonimiza nos dois níveis (Silver) e mede o recall contra o padrão-ouro (Gold). Só execute.
CREATE OR REPLACE TABLE silver.narrativas_anonimizadas
COMMENT 'Silver: narrativas com placas, telefones, documentos e nomes mascarados, nos níveis 1 e 2' AS
WITH sem_numeros AS (
  SELECT seq, texto,
         regexp_replace(regexp_replace(regexp_replace(texto,
           r'\b[A-Z]{3}-?\d[A-Z0-9]\d{2}\b', '[PLACA]'),
           r'\(?\d{2}\)?\s?9?\d{4}-?\d{4}', '[TELEFONE]'),
           r'\b\d{1,3}(\.\d{3}){2}-?[\dXx]{1,2}\b|\b\d{9,11}\b', '[DOCUMENTO]') AS t
  FROM bronze.narrativas_kdmile
)
SELECT seq, texto,
       -- nível 1: duas ou mais palavras seguidas com inicial maiúscula
       regexp_replace(t, r'(?U)\b[A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+(?:\s+(?:d[aeo]s?\s+)?[A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+)+', '[NOME]') AS nivel_1,
       -- nível 2: também palavras isoladas com inicial maiúscula fora do início de frase
       regexp_replace(t, r'(?U)(?<![.!?:]\s)(?<!^)\b[A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+(?:\s+(?:d[aeo]s?\s+)?[A-ZÁÉÍÓÚÂÊÔÃÕÇ][a-záéíóúâêôãõç]+)*', '[NOME]') AS nivel_2
FROM sem_numeros;

CREATE OR REPLACE TABLE gold.anonimizacao_recall
COMMENT 'Gold: quanto de cada tipo de dado pessoal cada nível removeu, contra o padrão-ouro do corpus' AS
WITH listas AS (
  SELECT seq, 'Nome' AS tipo, nomes AS lista FROM bronze.narrativas_kdmile
  UNION ALL SELECT seq, 'Localização', localizacoes FROM bronze.narrativas_kdmile
  UNION ALL SELECT seq, 'Documento', documentos FROM bronze.narrativas_kdmile
  UNION ALL SELECT seq, 'Placa', placas FROM bronze.narrativas_kdmile
  UNION ALL SELECT seq, 'Telefone', telefones FROM bronze.narrativas_kdmile
),
gabarito AS (
  SELECT seq, tipo, trim(e) AS entidade
  FROM listas LATERAL VIEW explode(split(lista, ';')) x AS e
  WHERE lista IS NOT NULL AND length(trim(e)) >= 2
),
niveis AS (
  SELECT seq, 1 AS nivel, nivel_1 AS anon FROM silver.narrativas_anonimizadas
  UNION ALL SELECT seq, 2, nivel_2 FROM silver.narrativas_anonimizadas
)
SELECT n.nivel, g.tipo, count(*) AS entidades_no_gabarito, count_if(instr(n.anon, g.entidade) > 0) AS vazaram,
       round(1 - count_if(instr(n.anon, g.entidade) > 0) / count(*), 3) AS removidas_recall
FROM gabarito g JOIN niveis n ON n.seq = g.seq
GROUP BY n.nivel, g.tipo;

SELECT a.tipo, a.entidades_no_gabarito, a.vazaram,
       concat(CAST(round(a.removidas_recall * 100) AS INT), '%') AS removidas_neste_nivel,
       concat(CAST(round(o.removidas_recall * 100) AS INT), '%') AS removidas_no_outro_nivel
FROM gold.anonimizacao_recall a
JOIN gold.lab2_anonimizacao_teste t ON a.nivel = t.nivel
LEFT JOIN gold.anonimizacao_recall o ON o.tipo = a.tipo AND o.nivel <> a.nivel
ORDER BY a.entidades_no_gabarito DESC

In [0]:
%sql
-- 🟢 PRONTA — antes e depois, numa narrativa. Só execute.
SELECT substr(a.texto, 1, 300) AS antes,
       substr(CASE WHEN t.nivel = 2 THEN a.nivel_2 ELSE a.nivel_1 END, 1, 300) AS depois
FROM silver.narrativas_anonimizadas a CROSS JOIN gold.lab2_anonimizacao_teste t
ORDER BY a.seq LIMIT 1

In [0]:
%sql
-- Atividade 4.2 — a decisão da Corregedoria.
CREATE OR REPLACE TABLE gold.lab2_decisao_anonimizacao COMMENT 'Lab 2: nível de anonimização escolhido e com quem compartilhar' AS
SELECT
  'TODO' AS nivel_escolhido,          -- '1' ou '2'
  'TODO' AS pode_compartilhar_com,    -- ex.: 'só a Inteligência', 'batalhões', 'ninguém fora da força'
  'TODO' AS justificativa             -- uma frase: o que ainda vaza e por que isso é aceitável (ou não)

**Para a plenária (11:35):** se o nível 2 remove 9 de cada 10 nomes, os que sobram justificam publicar? O que isso diz sobre o histórico do BO não ser público?

---

# Lab 2 (tarde) — Checagem de notícias e dashboard

## Parte 5 — A série do ISP da CISP do grupo (14:45–15:25)

As três primeiras células repetem o caminho do lab 1: baixar as séries públicas do ISP para o volume, criar a Bronze e padronizar na Silver (se o lab 1 rodou neste workspace, elas só atualizam as tabelas). Se o download falhar, entra o plano B embutido: dados reais das CISPs da RISP 1 e das CISPs 33 e 64, de jan/2024 a set/2026. A quarta célula monta a **Gold do grupo**: a série do recorte e o resumo do dashboard.

> **Atenção:** no ISP, os três meses mais recentes são **preliminares** (fase 2) e podem mudar. A população por CISP é a última estimativa do ISP (jul/2022).

In [0]:
# 🟢 PRONTA — baixa as séries públicas do ISPDados para o volume da Bronze. Só execute.
# O Databricks Free Edition limita o acesso à internet: se o site do ISP estiver bloqueado ou fora do ar,
# a célula grava no volume o PLANO B, uma amostra real do ISPDados embutida aqui (CISPs da RISP 1 e as
# CISPs 33 e 64, jan/2024 a set/2026, 14 delitos), baixada em 26/10/2026. Python só para o download; o resto é SQL.
import base64, gzip, urllib.request

VOLUME = "/Volumes/blue-blood/bronze/arquivos"
URLS = {
    "BaseDPEvolucaoMensalCisp.csv": "https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv",
    "PopulacaoEvolucaoMensalCisp.csv": "https://www.ispdados.rj.gov.br/Arquivos/PopulacaoEvolucaoMensalCisp.csv",
}
PLANO_B = {  # gzip + base64 dos dois arquivos, no mesmo formato do ISP (separador ;, latin-1)
    "BaseDPEvolucaoMensalCisp.csv": "...",
    "PopulacaoEvolucaoMensalCisp.csv": "H4sIAAAAAAACA03US64kRwgF0HmshUHwB9ViLOvJAw/sbtn7l3x5cpI9LApFAnGIrz//+fr89ce/n9///vH5+ePnb18IHP74R67Ih9ONnLOMjz1B1RIndb18/AnatQqkmtuJJ1gRl6mzLU5uplY7cXf76f2QhSIa0nb4bpRv4lPFVYe3qMQJSXUnVZ5gl2aT3BI+rG+q3yBJkTy8DbSLFLmmHt4GypKT3KQQzbcD1ku3Oftw7bESylSmF7m/9NDmQVUdR+4vI2xEucWObA9S7mFkrHlkmxBPDqH5mLw9pPqliEoca+9oUL5Sa/CRbYK1NZmsw4/EG42MJrs3j+65Ejd0Couj2y8G1R2kdv1ovVFtUaoWP/aWMHO95Cl6YqOWlSrkLQFHsUNQynu9x1Gso9tIxDThKNZRJIUHRhtPsPCbKfBfw1GsozTCwVFwFK8jSZr/xlGso0CumA6jWEYoivH9GEexjjBWLTUbR5sLHeQ4xMfRk+tcRlyXeSA9xaKBolsc35C2hVtN06sMpOdc8UYuLlMG0jbRIxzj7YH0znAaFsUlyHYBSAaL2hUDKRbSNUFUYyTFSmIjLBpmK/YOBxO81HM3sl2MJKyO56iLNxp5saatPZJiJWHxKYe4bsegVH2JcWtDaaNajFsvbK+9NQjgowvWb0qxlBxC4s6afXLHINgx1e8nKR9K2fBp7JCUK8kKRUEIJOVKEnznMoK5mYohIpg4svc7FrgcmKmBlC8kESxjykjKlcR4T/iyjKR8JAku0b0iR9LmGpauQkbH1g9J2lQi/S0pV5IW3sRr35K2BawBlSseVUjKlYT3BC9HDKR8IfH3NulAeieIqxpeMZByIQksjvyB9EQ9ehoG54H0fCxxtwj2ALV3OBrYPfQtAykXkhiWF3gH0kYjEO1KHki5kMAONeC9120YkLLw1P0PaaN46Zyk0Ju9NQDSFbpYioGUCwl5KBgP63+EPbVX8gYAAA==",
}

try:
    conteudos = {}
    for nome, url in URLS.items():
        with urllib.request.urlopen(url, timeout=60) as r:
            conteudos[nome] = r.read()
    ORIGEM_ISP = "ISPDados (download agora)"
except Exception as erro:
    conteudos = {nome: gzip.decompress(base64.b64decode(b)) for nome, b in PLANO_B.items()}
    ORIGEM_ISP = f"Plano B embutido (ISPDados de 26/10/2026); download falhou: {type(erro).__name__}"

spark.sql("""CREATE TABLE IF NOT EXISTS bronze.registro_ingestao
             (arquivo STRING, origem STRING, url STRING, bytes BIGINT, baixado_em TIMESTAMP)
             COMMENT 'Bronze: de onde veio cada arquivo e quando'""")
for nome, dados in conteudos.items():
    with open(f"{VOLUME}/{nome}", "wb") as f:
        f.write(dados)
    spark.sql("INSERT INTO bronze.registro_ingestao VALUES (:arquivo, :origem, :url, :bytes, current_timestamp())",
              args={"arquivo": nome, "origem": ORIGEM_ISP, "url": URLS[nome], "bytes": len(dados)})
print(ORIGEM_ISP)
print(f"Arquivos gravados em {VOLUME}: {', '.join(conteudos)}")

In [0]:
%sql
-- 🟢 PRONTA — Bronze: cada arquivo do volume vira uma tabela, sem alterar nenhum valor (tudo como texto). Só execute.
CREATE OR REPLACE TABLE bronze.isp_cisp_mensal
COMMENT 'Bronze: ISPDados BaseDPEvolucaoMensalCisp.csv como chegou (ocorrências por CISP e mês)'
AS SELECT *, current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/blue-blood/bronze/arquivos/BaseDPEvolucaoMensalCisp.csv',
                format => 'csv', header => true, sep => ';', encoding => 'ISO-8859-1', inferColumnTypes => false);

CREATE OR REPLACE TABLE bronze.isp_populacao_cisp
COMMENT 'Bronze: ISPDados PopulacaoEvolucaoMensalCisp.csv como chegou (população estimada por CISP)'
AS SELECT *, current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/blue-blood/bronze/arquivos/PopulacaoEvolucaoMensalCisp.csv',
                format => 'csv', header => true, sep => ';', inferColumnTypes => false);

SELECT 'bronze.isp_cisp_mensal' AS tabela, count(*) AS linhas FROM bronze.isp_cisp_mensal
UNION ALL
SELECT 'bronze.isp_populacao_cisp', count(*) FROM bronze.isp_populacao_cisp

In [0]:
%sql
-- 🟢 PRONTA — Silver: uma linha por CISP, mês e delito, com tipos certos. Só execute.
-- fase 2 = mês preliminar do ISP (ainda pode mudar); fase 3 = consolidado.
CREATE OR REPLACE TABLE silver.isp_ocorrencias
COMMENT 'Silver: ocorrências por CISP, mês e delito (ISPDados), tipadas; fase 2 = preliminar'
AS
SELECT cisp, aisp, risp, munic, make_date(ano, mes, 1) AS mes_ref, ano, mes, fase, delito, ocorrencias
FROM (
  SELECT CAST(cisp AS INT) AS cisp, CAST(aisp AS INT) AS aisp, CAST(risp AS INT) AS risp, munic,
         CAST(ano AS INT) AS ano, CAST(mes AS INT) AS mes, CAST(fase AS INT) AS fase,
         CAST(roubo_rua AS INT) AS roubo_rua,
         CAST(roubo_transeunte AS INT) AS roubo_transeunte,
         CAST(roubo_celular AS INT) AS roubo_celular,
         CAST(roubo_em_coletivo AS INT) AS roubo_em_coletivo,
         CAST(roubo_veiculo AS INT) AS roubo_veiculo,
         CAST(roubo_comercio AS INT) AS roubo_comercio,
         CAST(total_roubos AS INT) AS total_roubos,
         CAST(furto_transeunte AS INT) AS furto_transeunte,
         CAST(furto_celular AS INT) AS furto_celular,
         CAST(total_furtos AS INT) AS total_furtos,
         CAST(letalidade_violenta AS INT) AS letalidade_violenta,
         CAST(hom_doloso AS INT) AS hom_doloso,
         CAST(lesao_corp_dolosa AS INT) AS lesao_corp_dolosa,
         CAST(estupro AS INT) AS estupro
  FROM bronze.isp_cisp_mensal
)
UNPIVOT (ocorrencias FOR delito IN (roubo_rua, roubo_transeunte, roubo_celular, roubo_em_coletivo, roubo_veiculo, roubo_comercio, total_roubos, furto_transeunte, furto_celular, total_furtos, letalidade_violenta, hom_doloso, lesao_corp_dolosa, estupro))
WHERE cisp IS NOT NULL AND mes BETWEEN 1 AND 12;

CREATE OR REPLACE TABLE silver.isp_populacao
COMMENT 'Silver: última população estimada pelo ISP para cada CISP (a série vai até jul/2022)'
AS
SELECT CAST(circ AS INT) AS cisp,
       max_by(CAST(replace(pop_circ, ',', '.') AS DOUBLE), CAST(ano AS INT) * 100 + CAST(mes AS INT)) AS populacao,
       max(make_date(CAST(ano AS INT), CAST(mes AS INT), 1)) AS referencia
FROM bronze.isp_populacao_cisp
WHERE trim(pop_circ) <> ''
GROUP BY CAST(circ AS INT);

SELECT count(DISTINCT cisp) AS cisps, count(DISTINCT delito) AS delitos,
       min(mes_ref) AS primeiro_mes, max(mes_ref) AS ultimo_mes,
       count_if(fase = 2) AS linhas_preliminares
FROM silver.isp_ocorrencias

In [0]:
%sql
-- 🟢 PRONTA — Gold: a série do recorte do grupo e o resumo do dashboard. Só execute.
CREATE OR REPLACE TABLE gold.lab2_serie_recorte COMMENT 'Lab 2: últimos 24 meses do delito do grupo na CISP do grupo, com população' AS
SELECT o.cisp, o.delito, o.mes_ref, o.ocorrencias, o.fase, p.populacao
FROM silver.isp_ocorrencias o
JOIN gold.lab2_recorte r ON o.cisp = r.cisp AND o.delito = r.delito
LEFT JOIN silver.isp_populacao p ON p.cisp = o.cisp
WHERE o.mes_ref > add_months((SELECT max(mes_ref) FROM silver.isp_ocorrencias), -24);

CREATE OR REPLACE TABLE gold.lab2_resumo_isp COMMENT 'Lab 2: o resumo do dashboard (trimestres, 12 meses, taxa por 100 mil)' AS
WITH s AS (SELECT * FROM gold.lab2_serie_recorte),
u AS (SELECT max(CASE WHEN fase <> 2 THEN mes_ref END) AS ult, max(mes_ref) AS ult_qualquer FROM s),
a AS (                         -- somas por período; o trimestre de referência é o último DEFINITIVO
  SELECT max(s.cisp) AS cisp, max(s.delito) AS delito, max(u.ult) AS ult,
         sum(CASE WHEN s.mes_ref BETWEEN add_months(u.ult, -2) AND u.ult THEN s.ocorrencias END) AS tri_atual,
         sum(CASE WHEN s.mes_ref BETWEEN add_months(u.ult, -5) AND add_months(u.ult, -3) THEN s.ocorrencias END) AS tri_anterior,
         sum(CASE WHEN s.mes_ref BETWEEN add_months(u.ult, -14) AND add_months(u.ult, -12) THEN s.ocorrencias END) AS tri_ano_anterior,
         sum(CASE WHEN s.mes_ref > add_months(u.ult_qualquer, -12) THEN s.ocorrencias END) AS doze_meses,
         max(s.populacao) AS populacao
  FROM s CROSS JOIN u
),
taxas AS (                     -- taxa em 12 meses de todas as CISPs disponíveis, para comparar
  SELECT o.cisp, sum(o.ocorrencias) / max(p.populacao) * 100000 AS taxa
  FROM silver.isp_ocorrencias o JOIN silver.isp_populacao p ON p.cisp = o.cisp
  WHERE o.delito = (SELECT delito FROM gold.lab2_recorte)
    AND o.mes_ref > add_months((SELECT max(mes_ref) FROM silver.isp_ocorrencias), -12)
  GROUP BY o.cisp
)
SELECT a.cisp, a.delito,
       concat(date_format(add_months(a.ult, -2), 'yyyy-MM'), ' a ', date_format(a.ult, 'yyyy-MM')) AS ultimo_trimestre_definitivo,
       a.tri_atual AS ocorrencias_trimestre,
       round(try_divide(a.tri_atual - a.tri_anterior, a.tri_anterior), 3) AS variacao_vs_trimestre_anterior,
       round(try_divide(a.tri_atual - a.tri_ano_anterior, a.tri_ano_anterior), 3) AS variacao_vs_mesmo_trimestre_ano_anterior,
       a.doze_meses AS ocorrencias_12_meses, CAST(round(a.populacao) AS INT) AS populacao_estimada,
       round(a.doze_meses / a.populacao * 100000, 1) AS taxa_por_100_mil_12_meses,
       round(median(t.taxa), 1) AS mediana_taxa_das_cisps,
       count(t.cisp) AS cisps_comparadas,
       round(avg(CASE WHEN t.taxa < a.doze_meses / a.populacao * 100000 THEN 1.0 ELSE 0.0 END), 2) AS fracao_de_cisps_abaixo,
       CASE WHEN a.tri_atual IS NULL THEN 'A CISP ou o delito da Parte 0 não estão nos dados carregados. Troque na Parte 0 e rode de novo.'
            ELSE '' END AS aviso
FROM a CROSS JOIN taxas t
GROUP BY ALL;

SELECT * FROM gold.lab2_resumo_isp

In [0]:
# 🟢 PRONTA — o dashboard da CISP, lido da Gold. Só execute.
import matplotlib.pyplot as plt

serie = spark.table("gold.lab2_serie_recorte").orderBy("mes_ref").toPandas()
resumo = spark.table("gold.lab2_resumo_isp").first()
if serie.empty:
    raise ValueError("A série está vazia: a CISP ou o delito da Parte 0 não estão nos dados carregados. Troque na Parte 0 e rode de novo.")
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(serie["mes_ref"].astype(str).str[:7], serie["ocorrencias"],
       color=["#b23a3a" if f == 2 else "#0b2a4a" for f in serie["fase"]])
ax.set_title(f"CISP {resumo['cisp']} — {resumo['delito']} por mês (vermelho = preliminar)", loc="left")
ax.set_ylim(0, None)
ax.tick_params(axis="x", rotation=60, labelsize=8)
plt.tight_layout()
plt.show()

In [0]:
%sql
-- Atividade 5.1 — leitura do dashboard em 30 segundos.
CREATE OR REPLACE TABLE gold.lab2_leitura_dashboard COMMENT 'Lab 2: a leitura do dashboard pelo grupo' AS
SELECT
  'TODO' AS o_que,            -- indicador, delito, absoluto ou taxa
  'TODO' AS quando,           -- período e se há dado preliminar
  'TODO' AS comparado_com,    -- que base de comparação o grupo considera justa
  'TODO' AS e_dai             -- que decisão este número sustenta, com o limiar do lab 1

## Parte 6 — Julgar notícias com `ai_decide` (15:25–15:55)

O Databricks tem uma função SQL, [`ai_decide`](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_decide), que faz **julgamentos tipados**: os mesmos tipos do System One vistos na teoria (noul, choice, score), dentro do workspace e **sem chave de API**. Para cada notícia, a pergunta é um **noul** (sim ou não, com probabilidade): *"A afirmação da notícia é sustentada pela série do ISP no mesmo recorte e período?"*. A regra de decisão fica escrita **antes**, fora do modelo, numa tabela de parâmetros (`gold.lab2_regra_noticias`):

| Probabilidade | Decisão |
| --- | --- |
| 0,7 ou mais | Sustentada: responder confirmando, com o dado |
| de 0,4 a menos de 0,7 | Faixa ambígua: revisão humana antes de qualquer nota |
| abaixo de 0,4 | Não sustentada: contestar com o dado, período e fonte |

**Como as células rodam.** A notícia do grupo e as três notícias de exemplo sobre a CISP 12 vão para a tabela `gold.lab2_noticias_a_julgar`; o `ai_decide` julga cada linha e o resultado fica em `gold.lab2_julgamento_noticias`. A função está em *Beta* e só existe em algumas regiões: se o seu workspace não tiver `ai_decide`, a célula avisa e mostra resultados de exemplo **ilustrativos** (coerentes com os dados reais do ISP). Em qualquer caso, o grupo registra **antes** o próprio julgamento, para comparar com o da máquina.

In [0]:
%sql
-- Atividade 6.1 — a notícia que o grupo quer checar (Google News, sobre o recorte do grupo)
-- e o julgamento do grupo ANTES da IA.
CREATE OR REPLACE TABLE gold.lab2_noticia_do_grupo COMMENT 'Lab 2: a notícia do grupo e o julgamento humano registrado antes da IA' AS
SELECT
  'TODO' AS titulo,
  'TODO' AS fonte_e_data,
  'TODO' AS afirmacao,           -- a frase com o número que será checado
  'TODO' AS julgamento_humano;   -- a série sustenta? 'sim', 'não' ou 'em parte'

-- A regra de decisão sobre a probabilidade. O grupo pode mudar os limiares, mas só ANTES de rodar a IA.
CREATE OR REPLACE TABLE gold.lab2_regra_noticias COMMENT 'Lab 2: limiares de decisão sobre a probabilidade do ai_decide' AS
SELECT 0.7 AS sustentada_a_partir_de, 0.4 AS revisao_humana_a_partir_de

In [0]:
%sql
-- 🟢 PRONTA — as notícias a julgar: três exemplos sobre a CISP 12 e a do grupo, com a série do ISP. Só execute.
CREATE OR REPLACE TABLE gold.noticias_exemplo COMMENT 'Gold: notícias de exemplo sobre a CISP 12 e o que o ISP mostra' AS
SELECT * FROM VALUES
  ('Roubos de rua disparam 40% na CISP 12 no segundo trimestre', 'Alta de 40% em abr–jun/2026',
   'ISP: +30% contra jan–mar/26 e +16% contra abr–jun/25; puxado por um único mês (maio).', 0.46),
  ('Roubos de rua caem na CISP 12 no terceiro trimestre em relação a 2025', 'Queda em jul–set/2026 contra jul–set/2025',
   'ISP: 61 contra 84 (−27%), mas jul–set/26 ainda é preliminar.', 0.74),
  ('CISP 12 tem a maior taxa de roubos de rua do estado', 'Maior taxa por 100 mil do estado',
   'ISP: cerca de 352 por 100 mil em 12 meses; a CISP 5 passa de 5 mil.', 0.04)
AS t(titulo, afirmacao, serie_isp, p_exemplo);

CREATE OR REPLACE TABLE gold.lab2_noticias_a_julgar COMMENT 'Lab 2: estado de cada notícia (afirmação e série do ISP) para o ai_decide' AS
SELECT titulo, to_json(named_struct('noticia', afirmacao, 'serie_isp', serie_isp)) AS state, p_exemplo
FROM gold.noticias_exemplo
UNION ALL
SELECT n.titulo, to_json(named_struct('noticia', n.afirmacao, 'serie_isp', to_json(struct(r.*)))), CAST(NULL AS DOUBLE)
FROM gold.lab2_noticia_do_grupo n CROSS JOIN gold.lab2_resumo_isp r
WHERE n.titulo <> 'TODO' AND n.afirmacao <> 'TODO';

SELECT titulo, state FROM gold.lab2_noticias_a_julgar

In [0]:
# 🟢 PRONTA — julgamento tipado com ai_decide, em SQL. Só execute.
# O Python aqui só trata o caso de o workspace não ter ai_decide (Beta, só em algumas regiões):
# nesse caso a tabela recebe os resultados de exemplo ilustrativos.
import json

PERGUNTA = json.dumps({"evidencia_isp": {
    "type": "noul",
    "instructions": "A afirmação em noticia é sustentada pelos números em serie_isp, no mesmo recorte e período?",
    "criteria": {"true": "Os números do ISP confirmam a afirmação, no mesmo recorte e período",
                 "false": "Os números do ISP contradizem a afirmação ou não tratam do mesmo recorte e período"},
}}, ensure_ascii=False).replace("\\", "\\\\").replace("'", "\\'")

SQL_AI_DECIDE = f"""
CREATE OR REPLACE TABLE gold.lab2_julgamento_noticias COMMENT 'Lab 2: probabilidade de cada notícia ser sustentada pelo ISP' AS
SELECT titulo,
       variant_get(d, '$.response.answers.evidencia_isp.probability', 'double') AS p_sustentada,
       variant_get(d, '$.error_message', 'string') AS erro,
       'ai_decide' AS origem
FROM (SELECT titulo, ai_decide(state, '{PERGUNTA}') AS d FROM gold.lab2_noticias_a_julgar)
"""
SQL_EXEMPLO = """
CREATE OR REPLACE TABLE gold.lab2_julgamento_noticias COMMENT 'Lab 2: probabilidade de exemplo (ai_decide indisponível)' AS
SELECT titulo, p_exemplo AS p_sustentada, CAST(NULL AS STRING) AS erro,
       CASE WHEN p_exemplo IS NULL THEN '—' ELSE 'exemplo ilustrativo' END AS origem
FROM gold.lab2_noticias_a_julgar
"""
try:
    spark.sql(SQL_AI_DECIDE)
    erro = spark.sql("SELECT max(erro) FROM gold.lab2_julgamento_noticias").first()[0]
    if erro:
        raise RuntimeError(erro)
    print("ai_decide respondeu no seu workspace.")
except Exception as e:
    spark.sql(SQL_EXEMPLO)
    print(f"ai_decide indisponível neste workspace ({type(e).__name__}): usando resultados de exemplo ilustrativos.")

In [0]:
%sql
-- 🟢 PRONTA — aplica a regra de decisão da tabela gold.lab2_regra_noticias. Só execute.
CREATE OR REPLACE TABLE gold.lab2_decisao_noticias COMMENT 'Lab 2: decisão sobre cada notícia, pela regra escrita antes da IA' AS
SELECT j.titulo AS noticia, round(j.p_sustentada, 2) AS p_sustentada,
       CASE WHEN j.p_sustentada IS NULL THEN 'sem ai_decide: vale o julgamento humano'
            WHEN j.p_sustentada >= r.sustentada_a_partir_de THEN 'SUSTENTADA: confirmar com o dado'
            WHEN j.p_sustentada >= r.revisao_humana_a_partir_de THEN 'AMBÍGUA: revisão humana'
            ELSE 'NÃO SUSTENTADA: contestar com dado, período e fonte' END AS decisao,
       j.origem,
       coalesce(e.serie_isp, 'resumo do ISP do grupo (gold.lab2_resumo_isp)') AS o_que_o_isp_mostra,
       g.julgamento_humano AS julgamento_humano_do_grupo
FROM gold.lab2_julgamento_noticias j
CROSS JOIN gold.lab2_regra_noticias r
LEFT JOIN gold.noticias_exemplo e ON e.titulo = j.titulo
LEFT JOIN gold.lab2_noticia_do_grupo g ON g.titulo = j.titulo;

SELECT * FROM gold.lab2_decisao_noticias

**Para discutir:** a notícia 1 tem um número real por trás (+30%), mas não o da manchete (+40%), e cai na faixa ambígua. O que a revisão humana precisa olhar? A probabilidade do `ai_decide` concorda com o julgamento humano do grupo? Em que casos o grupo aceitaria agir com probabilidade abaixo de 0,7? (Lembre do slide "o limiar nasce do custo de errar".)

---

## Parte 7 — O que levar para a aula 3 (15:55–16:20)

Na aula 3 (assíncrona) o grupo ensaia o arco inteiro sobre **uma** notícia. Escolham agora.

In [0]:
%sql
-- Parte 7 — escolhas para o ensaio da aula 3.
CREATE OR REPLACE TABLE gold.lab2_ensaio_aula3 COMMENT 'Lab 2: a notícia e a comparação escolhidas para o ensaio da aula 3' AS
SELECT
  'TODO' AS noticia,                    -- título e fonte da notícia que o grupo vai checar
  'TODO' AS serie_isp,                  -- ex.: 'CISP 12, roubo_rua, out/2024–set/2026'
  'TODO' AS base_de_comparacao,         -- ex.: 'mesmo trimestre do ano anterior'
  'TODO' AS absoluto_ou_taxa,           -- 'absoluto', 'taxa por 100 mil' ou 'os dois'
  'TODO' AS ideia_de_politica_publica;  -- um rascunho: público, ação e indicador

-- Rodada 2 da Operação Manchete (síntese das 16:30). Troque só a palavra NULL pelos pontos.
CREATE OR REPLACE TABLE gold.lab2_rodada2 COMMENT 'Lab 2: porta e pontos da Rodada 2 da Operação Manchete' AS
SELECT
  'TODO' AS porta,                      -- 'A', 'B', 'C', 'D' ou 'E'
  CAST(NULL AS INT) AS pontos           -- 0 a 3, anunciado pelo professor

## Parte 8 — Checagem final (16:20–16:30)

Rode as duas células: a primeira lista o que falta; a segunda monta o resumo para o fórum, lido das tabelas do grupo.

In [0]:
%sql
-- 🟢 PRONTA — confere o que falta. Só execute.
-- Se aparecer TABLE_OR_VIEW_NOT_FOUND, a célula daquela parte ainda não foi executada.
SELECT item, CASE WHEN coalesce(ok, false) THEN 'OK' ELSE 'FALTA' END AS situacao
FROM (
  SELECT 1, 'Parte 0 — número do grupo (1 a 8)', (SELECT max(grupo) BETWEEN 1 AND 8 FROM gold.lab2_recorte)
  UNION ALL SELECT 2, 'Parte 2 — auditoria das 10 narrativas', (SELECT count_if(avaliacao IN ('certo', 'errado', 'incompleto')) = count(*) FROM gold.lab2_auditoria)
  UNION ALL SELECT 3, 'Parte 3 — nome de cada grupo de texto e o grupo mais útil', (SELECT count_if(n.nome IS NULL OR n.nome = 'TODO') = 0 FROM gold.grupos_de_texto g LEFT JOIN gold.lab2_nomes_grupos n ON n.grupo = g.grupo) AND (SELECT count(*) > 0 AND count_if(to_json(struct(*)) LIKE '%TODO%') = 0 FROM gold.lab2_grupo_mais_util)
  UNION ALL SELECT 4, 'Parte 4 — decisão de anonimização', (SELECT count(*) > 0 AND count_if(to_json(struct(*)) LIKE '%TODO%') = 0 FROM gold.lab2_decisao_anonimizacao)
  UNION ALL SELECT 5, 'Parte 5 — série do ISP na Gold', (SELECT count(*) > 0 FROM gold.lab2_serie_recorte)
  UNION ALL SELECT 6, 'Parte 5 — leitura do dashboard', (SELECT count(*) > 0 AND count_if(to_json(struct(*)) LIKE '%TODO%') = 0 FROM gold.lab2_leitura_dashboard)
  UNION ALL SELECT 7, 'Parte 6 — notícia do grupo e julgamento humano', (SELECT count(*) > 0 AND count_if(to_json(struct(*)) LIKE '%TODO%') = 0 FROM gold.lab2_noticia_do_grupo)
  UNION ALL SELECT 8, 'Parte 7 — escolhas para a aula 3', (SELECT count(*) > 0 AND count_if(to_json(struct(*)) LIKE '%TODO%') = 0 FROM gold.lab2_ensaio_aula3)
  UNION ALL SELECT 9, 'Parte 7 — Rodada 2 registrada', (SELECT max(porta IN ('A', 'B', 'C', 'D', 'E') AND pontos BETWEEN 0 AND 3) FROM gold.lab2_rodada2)
) AS t(ordem, item, ok)
ORDER BY ordem

In [0]:
%sql
-- 🟢 PRONTA — resumo para o fórum da aula 3, lido das tabelas do grupo. Só execute.
SELECT 1 AS ordem, concat('Grupo ', grupo, ' — CISP ', cisp, ', ', delito) AS resumo FROM gold.lab2_recorte
UNION ALL
SELECT 2, concat('ISP: ', ocorrencias_12_meses, ' ocorrências em 12 meses; ', taxa_por_100_mil_12_meses, ' por 100 mil')
FROM gold.lab2_resumo_isp
UNION ALL
SELECT 3, concat('Notícia para o ensaio: ', noticia) FROM gold.lab2_ensaio_aula3
UNION ALL
SELECT 4, concat('Base de comparação: ', base_de_comparacao, ' | ', absoluto_ou_taxa) FROM gold.lab2_ensaio_aula3
UNION ALL
SELECT 5, concat('Anonimização: nível ', nivel_escolhido, ', compartilhar com ', pode_compartilhar_com)
FROM gold.lab2_decisao_anonimizacao
ORDER BY ordem

## Material de apoio

- **Slides da aula 2**: arquitetura medalhão, modelos de linguagem, NLP em boletins, LAI e LGPD, julgamentos com probabilidade, calibração, "40% sobre quanto?", taxa por 100 mil, dashboards e política pública.
- ALVES, D. et al. Classificação de Boletins de Ocorrências através de Modelos de Linguagem baseados em BERT. LASDiGov/WCGE, SBC, 2024.
- ALMEIDA, M. G. de; BOTEGA, L. C.; LIMA, P. M. de P. Local Anonymization of Free-Text Robbery Police Reports in Brazilian Portuguese. KDMiLe, SBC, 2026 (dataset CC BY 4.0).
- [`ai_decide` — documentação do Databricks](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_decide) (Beta); o padrão de julgamentos tipados vem do [TypeSafe System One](https://docs.typesafe.ai/concepts/how-to-build-with-system-one), visto na teoria.
- [ISPDados](https://www.ispdados.rj.gov.br/) — séries por CISP e população.

**O que este lab deixa no espaço `blue-blood`:** na Bronze, as narrativas e as séries do ISP como chegaram; na Silver, as narrativas classificadas, agrupadas e anonimizadas, e a série padronizada; na Gold, o resumo do recorte, o recall da anonimização, o julgamento das notícias e todas as tabelas `lab2_` com as escolhas do grupo.

**Na aula 3 (11/11, assíncrona):** leituras básicas e ensaio do arco completo sobre a notícia escolhida; teste de acesso ao ISP Microdados; Rodada 3 no fórum.